# Dora Maar — Artistic Legacy & Online Findability

This notebook builds a best-effort catalog of Dora Maar's known works from public APIs, then measures how findable each work is online via a composite impressions proxy (Google Trends + Wikipedia pageviews).

**Caveats:**
- No complete public catalog raisonné exists; the 2019 Centre Pompidou retrospective (500+ works) is print-only.
- Picasso's portraits *of* Dora Maar are excluded — they are his works, not hers.
- Google Trends scores are relative (max in period = 100), not absolute impression counts.
- Sources: Wikidata SPARQL, Met Museum API, Wikimedia Commons MediaWiki API.

In [1]:
import os, ssl, certifi

# macOS Python 3.13 from python.org ships without system certs — point it at certifi
os.environ["SSL_CERT_FILE"]      = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import time
import re
import requests
import pandas as pd
import plotly.express as px
from SPARQLWrapper import SPARQLWrapper, JSON
from pytrends.request import TrendReq
from datetime import date, timedelta

HEADERS = {"User-Agent": "DoraMaarResearch/1.0 (research; pietro@datatecnica.com)"}

---
## Section 1 — Full Catalog

Assembled from three sources:
1. **Wikidata SPARQL** — works where `P170` (creator) = `Q236161` (Dora Maar)
2. **Met Museum API** — search by artist name
3. **Wikimedia Commons** — `Category:Photographs_by_Dora_Maar`

In [2]:
# --- 1a. Wikidata SPARQL ---
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", HEADERS["User-Agent"])
sparql.setQuery("""
SELECT DISTINCT ?item ?itemLabel ?year ?mediumLabel ?institutionLabel ?image WHERE {
  ?item wdt:P170 wd:Q236161 .
  OPTIONAL { ?item wdt:P571 ?created . BIND(YEAR(?created) AS ?year) }
  OPTIONAL { ?item wdt:P186 ?medium }
  OPTIONAL { ?item wdt:P276 ?institution }
  OPTIONAL { ?item wdt:P18  ?image }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,fr" }
}
ORDER BY ?year
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

wikidata_rows = []
for r in results["results"]["bindings"]:
    wikidata_rows.append({
        "title":       r.get("itemLabel", {}).get("value", ""),
        "year":        r.get("year",      {}).get("value", ""),
        "medium":      r.get("mediumLabel",{}).get("value", ""),
        "institution": r.get("institutionLabel", {}).get("value", ""),
        "image_url":   r.get("image",     {}).get("value", ""),
        "source":      "Wikidata",
    })

df_wikidata = pd.DataFrame(wikidata_rows)
print(f"Wikidata: {len(df_wikidata)} records")
df_wikidata.head()

Wikidata: 29 records


,title,year,medium,institution,image_url,source
0,Sky Pinwheels and Bottles,,,Cleveland Museum of Art,,Wikidata
1,Crowd in Front of a Car,,,Cleveland Museum of Art,,Wikidata
2,Three Covered Statues,,,Cleveland Museum of Art,,Wikidata
3,Couple Kissing,,,Cleveland Museum of Art,,Wikidata
4,Double Portrait with Hat,,,Cleveland Museum of Art,,Wikidata


In [3]:
# --- 1b. Met Museum API ---
base = "https://collectionapi.metmuseum.org/public/collection/v1"

# artistOrCulture=true returns 0 for "Dora Maar" — use plain keyword search and filter by artistDisplayName
search_resp = requests.get(
    f"{base}/search",
    params={"q": "Dora Maar"},
    headers=HEADERS,
).json()

object_ids = search_resp.get("objectIDs") or []
print(f"Met search returned {len(object_ids)} object IDs")

met_rows = []
for oid in object_ids:
    obj = requests.get(f"{base}/objects/{oid}", headers=HEADERS).json()
    artist = obj.get("artistDisplayName", "")
    if artist.lower() != "dora maar":
        continue
    met_rows.append({
        "title":       obj.get("title", ""),
        "year":        obj.get("objectDate", ""),
        "medium":      obj.get("medium", ""),
        "institution": "The Metropolitan Museum of Art",
        "image_url":   obj.get("primaryImageSmall", ""),
        "source":      "Met Museum",
    })
    time.sleep(0.1)

df_met = pd.DataFrame(met_rows)
print(f"Met Museum: {len(df_met)} works by Dora Maar")
df_met.head()

Met search returned 23 object IDs
Met Museum: 3 works by Dora Maar


,title,year,medium,institution,image_url,source
0,[Boy with a Cat],1934,Gelatin silver print,The Metropolitan Museum of Art,,Met Museum
1,Père Ubu,1936,Gelatin silver print,The Metropolitan Museum of Art,,Met Museum
2,Chanteurs unijambistes,1934,Gelatin silver print,The Metropolitan Museum of Art,,Met Museum


In [4]:
# --- 1c. Wikimedia Commons: file search for "Dora Maar" ---
# Category:Photographs by Dora Maar does not exist; use full-text search in File namespace (ns=6)
wiki_api = "https://commons.wikimedia.org/w/api.php"
commons_rows = []
gsroffset = 0

while True:
    params = {
        "action":      "query",
        "generator":   "search",
        "gsrsearch":   "Dora Maar",
        "gsrnamespace":"6",
        "gsrlimit":    "50",
        "gsroffset":   str(gsroffset),
        "prop":        "imageinfo",
        "iiprop":      "url",
        "format":      "json",
    }
    resp = requests.get(wiki_api, params=params, headers=HEADERS).json()
    pages = resp.get("query", {}).get("pages", {})
    for page in pages.values():
        raw_title = page.get("title", "")
        title = re.sub(r"^File:", "", raw_title)
        title = re.sub(r"\.[a-zA-Z]{2,5}$", "", title).strip()
        commons_rows.append({
            "title":       title,
            "year":        "",
            "medium":      "Photography",
            "institution": "Wikimedia Commons",
            "image_url":   (page.get("imageinfo") or [{}])[0].get("url", ""),
            "source":      "Wikimedia Commons",
        })
    if "continue" in resp:
        gsroffset = resp["continue"].get("gsroffset", gsroffset + 50)
    else:
        break

df_commons = pd.DataFrame(commons_rows)
print(f"Wikimedia Commons: {len(df_commons)} files found")
df_commons.head()

Wikimedia Commons: 3239 files found


,title,year,medium,institution,image_url,source
0,Accès Gare Saint Ouen Rue Dora Maar - Saint-Ou...,,Photography,Wikimedia Commons,https://upload.wikimedia.org/wikipedia/commons...,Wikimedia Commons
1,Accès Gare Saint Ouen Rue Dora Maar - Saint-Ou...,,Photography,Wikimedia Commons,https://upload.wikimedia.org/wikipedia/commons...,Wikimedia Commons
2,Accès Gare Saint Ouen Rue Dora Maar - Saint-Ou...,,Photography,Wikimedia Commons,https://upload.wikimedia.org/wikipedia/commons...,Wikimedia Commons
3,Accès Gare Saint Ouen Rue Dora Maar - Saint-Ou...,,Photography,Wikimedia Commons,https://upload.wikimedia.org/wikipedia/commons...,Wikimedia Commons
4,Accès Gare Saint Ouen Rue Dora Maar - Saint-Ou...,,Photography,Wikimedia Commons,https://upload.wikimedia.org/wikipedia/commons...,Wikimedia Commons


In [5]:
# --- 1d. Combine & deduplicate ---
df_catalog = pd.concat([df_wikidata, df_met, df_commons], ignore_index=True)

def normalize_title(t):
    t = str(t).lower().strip()
    t = re.sub(r"[^a-z0-9\s]", "", t)
    t = re.sub(r"\s+", " ", t)
    return t

df_catalog["title_norm"] = df_catalog["title"].apply(normalize_title)

# keep first occurrence per normalized title (prioritize Wikidata > Met > Commons)
df_catalog = df_catalog.drop_duplicates(subset="title_norm", keep="first").reset_index(drop=True)

# drop rows with blank or Q-code titles (Wikidata items without English labels)
df_catalog = df_catalog[
    (df_catalog["title"].str.len() > 2)
    & (~df_catalog["title"].str.match(r"^Q\d+$"))
].reset_index(drop=True)

print(f"\nFinal catalog: {len(df_catalog)} unique works")
print("\nSources breakdown:")
print(df_catalog["source"].value_counts().to_string())

# Verification: known works must be present
known = ["père ubu", "shell", "simulateur", "rue d astorg"]
for kw in known:
    hits = df_catalog[df_catalog["title_norm"].str.contains(kw, na=False)]
    status = "✓" if len(hits) else "✗ MISSING"
    print(f"  {status}  '{kw}' → {list(hits['title'].head(2))}")

df_catalog[["title", "year", "medium", "institution", "source"]].head(10)


Final catalog: 3195 unique works

Sources breakdown:
source
Wikimedia Commons    3167
Wikidata               28
  ✗ MISSING  'père ubu' → []
  ✓  'shell' → ['Alabaster & shell head of a woman, Early Dynastic Period III, c. 2400 BC (27682648643)']
  ✗ MISSING  'simulateur' → []
  ✗ MISSING  'rue d astorg' → []


,title,year,medium,institution,source
0,Sky Pinwheels and Bottles,,,Cleveland Museum of Art,Wikidata
1,Crowd in Front of a Car,,,Cleveland Museum of Art,Wikidata
2,Three Covered Statues,,,Cleveland Museum of Art,Wikidata
3,Couple Kissing,,,Cleveland Museum of Art,Wikidata
4,Double Portrait with Hat,,,Cleveland Museum of Art,Wikidata
5,Trees and Ominous Sky,,,Cleveland Museum of Art,Wikidata
6,Carousel at Night,,,Cleveland Museum of Art,Wikidata
7,Portrait of Nush Eluard,1920,gelatin silver print,Israel Museum,Wikidata
8,Christian Berard's Head Posed at Edge of Basin...,1930,gelatin silver print,National Gallery of Art,Wikidata
9,The Grimace,1933,,Cleveland Museum of Art,Wikidata


---
## Section 2 — Google-Visible Artworks

We query the Wikipedia API for Dora Maar-related article text and the Wikidata labels already fetched, then cross-reference which catalog titles appear in the top Wikipedia result and in Wikidata descriptions. This is a proxy for "what Google surfaces" without requiring a paid search API.

In [6]:
wp_api = "https://en.wikipedia.org/w/api.php"

def wp_article_text(title, intro_only=False):
    params = {
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
        "format": "json",
    }
    if intro_only:
        params["exintro"] = True
    resp = requests.get(wp_api, params=params, headers=HEADERS).json()
    pages = resp.get("query", {}).get("pages", {})
    return next(iter(pages.values()), {}).get("extract", "")

def has_wp_page(title):
    resp = requests.get(wp_api, params={
        "action": "query",
        "titles": title,
        "format": "json",
    }, headers=HEADERS).json()
    pages = resp.get("query", {}).get("pages", {})
    return "-1" not in pages

# Full text of the main Dora Maar article (needed for title-in-text check)
dora_maar_text = wp_article_text("Dora Maar")
print(f"Wikipedia 'Dora Maar' article length: {len(dora_maar_text):,} chars")

# For each catalog work, check if it has its own Wikipedia article that mentions Dora Maar.
# Using intro_only=True for speed; "Forbidden Games" (film) and "Children Playing" etc.
# will NOT mention Dora Maar and are therefore excluded.
print("Checking Wikipedia articles for each work...")
wp_maar_pages = {}  # title -> True if WP article exists AND mentions "dora maar"
for title in df_catalog["title"].unique():
    if has_wp_page(title):
        text = wp_article_text(title, intro_only=True)
        wp_maar_pages[title] = "dora maar" in text.lower()
    time.sleep(0.15)

df_catalog["in_wikipedia_article"] = df_catalog["title"].apply(
    lambda t: t.lower() in dora_maar_text.lower()
)
df_catalog["has_maar_wp_page"] = df_catalog["title"].map(wp_maar_pages).fillna(False)

# Google-visible = title appears in main Dora Maar article OR has its own WP article about the work
df_catalog["google_visible"] = df_catalog["in_wikipedia_article"] | df_catalog["has_maar_wp_page"]

df_visible = df_catalog[df_catalog["google_visible"]].copy()
print(f"\nGoogle-visible artworks: {len(df_visible)}")
df_visible[["title", "year", "source", "in_wikipedia_article", "has_maar_wp_page"]]

Wikipedia 'Dora Maar' article length: 13,749 chars
Checking Wikipedia articles for each work...

Google-visible artworks: 0


,title,year,source,in_wikipedia_article,has_maar_wp_page


---
## Section 3 — Impressions Score per Artwork

**Method:**
- **Google Trends** (`pytrends`): query each artwork title paired with "Dora Maar"; take mean relative interest (0–100) over the past 12 months. Batched 5 per call with retry on 429.
- **Wikipedia pageviews**: 7-day view count via Wikimedia REST API for artworks that have a Wikipedia page.
- **Combined score**: `trends_score × 0.7 + normalized_pageviews × 0.3`

In [7]:
# --- 3a. Google Trends scoring ---
# Only score Google-visible works (trends for unknown works return 0 / noise)
score_titles = df_visible["title"].dropna().unique().tolist()
# Prepend "Dora Maar" to each query to anchor context
kw_map = {t: f"Dora Maar {t}"[:100] for t in score_titles}  # pytrends caps at 100 chars

def trends_batch(keywords, retries=3):
    """Return {keyword: mean_score} for a batch of ≤5 keywords."""
    pt = TrendReq(hl="en-US", tz=0, timeout=(10, 30))
    for attempt in range(retries):
        try:
            pt.build_payload(keywords, timeframe="today 12-m", geo="")
            df_t = pt.interest_over_time()
            if df_t.empty:
                return {k: 0.0 for k in keywords}
            return {k: float(df_t[k].mean()) for k in keywords if k in df_t.columns}
        except Exception as e:
            wait = 60 * (attempt + 1)
            print(f"  Trends error (attempt {attempt+1}): {e} — waiting {wait}s")
            time.sleep(wait)
    return {k: 0.0 for k in keywords}

# Batch into groups of 5
kw_list = list(kw_map.values())
trends_scores = {}
for i in range(0, len(kw_list), 5):
    batch = kw_list[i:i+5]
    print(f"Trends batch {i//5 + 1}/{(len(kw_list)-1)//5 + 1}: {batch}")
    scores = trends_batch(batch)
    trends_scores.update(scores)
    time.sleep(2)  # be polite

# Map back from anchored keyword → original title
rev_kw_map = {v: k for k, v in kw_map.items()}
df_visible["trends_score"] = df_visible["title"].map(
    lambda t: trends_scores.get(kw_map.get(t, ""), 0.0)
)
print(f"\nTrends scores assigned. Non-zero: {(df_visible['trends_score'] > 0).sum()}")
df_visible[["title", "trends_score"]].sort_values("trends_score", ascending=False).head(10)


Trends scores assigned. Non-zero: 0


,title,trends_score


In [8]:
# --- 3b. Wikipedia pageviews (7-day) ---
# Only counts views for articles confirmed to be about this Dora Maar work (from wp_maar_pages).
# This prevents false positives like "Forbidden Games" (film) or "Père Ubu" (Jarry character).
end_date   = date.today() - timedelta(days=1)
start_date = end_date - timedelta(days=6)
pv_base = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"

def get_pageviews(article_title):
    safe = requests.utils.quote(article_title.replace(" ", "_"), safe="")
    url = f"{pv_base}/en.wikipedia/all-access/all-agents/{safe}/daily/{start_date:%Y%m%d}/{end_date:%Y%m%d}"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10).json()
        items = resp.get("items", [])
        return sum(i["views"] for i in items)
    except Exception:
        return 0

print("Fetching pageviews for verified Dora Maar Wikipedia articles...")
pv_data = {}
for title in df_visible["title"].unique():
    if wp_maar_pages.get(title, False):  # only articles confirmed to be about the artwork
        pv = get_pageviews(title)
        if pv > 0:
            pv_data[title] = pv
            print(f"  {title}: {pv:,} views")

df_visible["pageviews_7d"] = df_visible["title"].map(pv_data).fillna(0)

max_pv = df_visible["pageviews_7d"].max()
df_visible["pageviews_norm"] = (
    (df_visible["pageviews_7d"] / max_pv * 100) if max_pv > 0 else 0
)
print(f"\nPageview data for {len(pv_data)} works.")

Fetching pageviews for verified Dora Maar Wikipedia articles...

Pageview data for 0 works.


In [9]:
# --- 3c. Combined impressions score ---
df_visible["impressions_score"] = (
    df_visible["trends_score"]  * 0.7
    + df_visible["pageviews_norm"] * 0.3
).round(2)

print("Impressions score distribution:")
print(df_visible["impressions_score"].describe().round(2).to_string())
df_visible[["title", "year", "trends_score", "pageviews_7d", "pageviews_norm", "impressions_score"]] \
    .sort_values("impressions_score", ascending=False).head(15)

Impressions score distribution:
count       0
unique      0
top       NaN
freq      NaN


,title,year,trends_score,pageviews_7d,pageviews_norm,impressions_score


---
## Section 4 — Ranked Table & Visualization

Top artworks by impressions score (Google Trends + Wikipedia pageviews proxy).

In [10]:
# --- 4a. Full ranked table ---
df_ranked = (
    df_visible[["title", "year", "medium", "institution", "source",
                "trends_score", "pageviews_7d", "impressions_score"]]
    .sort_values("impressions_score", ascending=False)
    .reset_index(drop=True)
)
df_ranked.index += 1  # rank starts at 1
df_ranked.index.name = "rank"

print(f"Ranked catalog: {len(df_ranked)} works")
df_ranked.head(30)

Ranked catalog: 0 works


,title,year,medium,institution,source,trends_score,pageviews_7d,impressions_score
rank,,,,,,,,


In [11]:
# --- 4b. Bar chart: top 30 artworks by impressions score ---
top30 = df_ranked.head(30).reset_index()

fig = px.bar(
    top30,
    x="impressions_score",
    y="title",
    orientation="h",
    color="source",
    hover_data=["year", "medium", "trends_score", "pageviews_7d"],
    title="Dora Maar — Top 30 Works by Online Impressions Score",
    labels={
        "impressions_score": "Impressions Score (0–100)",
        "title": "",
        "source": "Data Source",
    },
    color_discrete_map={
        "Wikidata":         "#4C78A8",
        "Met Museum":       "#E45756",
        "Wikimedia Commons":"#72B7B2",
    },
    height=750,
)
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    plot_bgcolor="white",
    font_family="Georgia",
)
fig.show()

In [12]:
# --- Summary statistics ---
print("=== SUMMARY ===")
print(f"Total catalog works  : {len(df_catalog)}")
print(f"Google-visible works : {len(df_visible)}")
print(f"Works with Trends > 0: {(df_visible['trends_score'] > 0).sum()}")
print(f"Works with pageviews : {(df_visible['pageviews_7d'] > 0).sum()}")
print()
print("Top 5 most findable works:")
for i, row in df_ranked.head(5).iterrows():
    print(f"  {i}. {row['title']} ({row['year']}) — score: {row['impressions_score']}")

=== SUMMARY ===
Total catalog works  : 3195
Google-visible works : 0
Works with Trends > 0: 0
Works with pageviews : 0

Top 5 most findable works:
